## Data Loading

In [197]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"


CPU times: total: 0 ns
Wall time: 0 ns


In [198]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3


# Training with  XGBoost (Extreme Gradient Boosting)



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [266]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(
    cat_df,
    tie_tolerance=0.0  # algorithms within tie_tolerance% of min treated as ties
)
# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Expanded 187 rows with near ties using tolerance 0.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [267]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Train XGBoost with sample weights
xgb_strat_y = XGBClassifier(
     objective="multi:softprob",
     num_class=len(algorithms),      # = 5 in your case
     n_estimators=400,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     subsample=0.8,
     use_label_encoder=False,
     random_state=42
 )

xgb_strat_y.fit(X_train, y_train, sample_weight=weights_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_class=5, ...)

Feature Importances

In [268]:
# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_strat_y.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
4           d_max    0.161008
2               m    0.138510
10  shell_minSize    0.110289
6        core_min    0.100352
11  shell_stdSize    0.098008
5        core_max    0.096857
7       shell_num    0.077169
3           d_avg    0.075417
0         k_value    0.054662
8   shell_avgSize    0.053396
1               n    0.034332
9   shell_maxSize    0.000000


#### Inference on 20% dataset

In [212]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=xgb_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./XGB_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./XGB_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 59.89%

Precision by Algorithm:
  UMKP: 59.38% (32 predictions)
  AltRB: 54.17% (24 predictions)
  kPEX: 67.92% (106 predictions)
  Pivot_Br: 42.16% (102 predictions)
  S_Br: 72.22% (90 predictions)

Recall by Algorithm:
  UMKP: 64.58% (31/48 cases)
  AltRB: 53.19% (25/47 cases)
  kPEX: 65.57% (120/183 cases)
  Pivot_Br: 69.05% (87/126 cases)
  S_Br: 78.83% (108/137 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 37378.30
  Total Optimal Time   = 35466.00
  Time Ratio (pred/opt)= 1.054
  Avg Overhead/case ((pred - opt) / opt)    = 32.87%


Adding prediction probabilities...

Saving full predictions to './XGB_results/tie_tol/50/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 142 wrong predictions with time metrics

Top 5 heav

#### Re-evalute on entire dataset

In [213]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=xgb_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./XGB_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./XGB_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 64.74%

Precision by Algorithm:
  UMKP: 60.29% (68 predictions)
  AltRB: 66.67% (87 predictions)
  kPEX: 72.65% (234 predictions)
  Pivot_Br: 42.78% (194 predictions)
  S_Br: 79.10% (177 predictions)

Recall by Algorithm:
  UMKP: 66.00% (66/100 cases)
  AltRB: 66.38% (77/116 cases)
  kPEX: 68.32% (248/363 cases)
  Pivot_Br: 72.57% (164/226 cases)
  S_Br: 81.64% (209/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 69267.40
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.036
  Avg Overhead/case ((pred - opt) / opt)    = 26.19%


Adding prediction probabilities...

Saving full predictions to './XGB_results/tie_tol/50/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 268 wrong predictions with time metrics

Top

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [269]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(
    cat_df,
    test_size = 0.2,
    tie_tolerance=0.0  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Expanded 187 rows with near ties using tolerance 0.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [270]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Train XGBoost with sample weights
xgb_strat_y_K = XGBClassifier(
     objective="multi:softprob",
     num_class=len(algorithms),      # = 5 in our case
     n_estimators=400,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     subsample=0.8,
     random_state=42
 )

xgb_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_class=5, ...)

Feature Importances

In [271]:
# --- Feature Importance ---
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_strat_y_K.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
4           d_max    0.149100
2               m    0.141427
6        core_min    0.111182
5        core_max    0.110270
11  shell_stdSize    0.109181
10  shell_minSize    0.081769
3           d_avg    0.078836
0         k_value    0.060405
1               n    0.057679
7       shell_num    0.057152
8   shell_avgSize    0.042999
9   shell_maxSize    0.000000


#### Inference on 20% dataset

In [231]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=xgb_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./XGB_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./XGB_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.00%

Precision by Algorithm:
  UMKP: 30.77% (39 predictions)
  AltRB: 57.78% (45 predictions)
  kPEX: 63.64% (132 predictions)
  Pivot_Br: 37.39% (115 predictions)
  S_Br: 50.56% (89 predictions)

Recall by Algorithm:
  UMKP: 69.81% (37/53 cases)
  AltRB: 57.97% (40/69 cases)
  kPEX: 57.85% (129/223 cases)
  Pivot_Br: 73.55% (114/155 cases)
  S_Br: 64.38% (94/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 54172.30
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.167
  Avg Overhead/case ((pred - opt) / opt)    = 12.92%


Adding prediction probabilities...

Saving full predictions to './XGB_results/tie_tol/50/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 210 wrong predictions with time metrics

Top 5 he

#### Re-evalute on entire dataset

In [232]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=xgb_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./XGB_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./XGB_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 54.34%

Precision by Algorithm:
  UMKP: 40.91% (66 predictions)
  AltRB: 57.14% (91 predictions)
  kPEX: 67.30% (211 predictions)
  Pivot_Br: 37.50% (240 predictions)
  S_Br: 67.11% (152 predictions)

Recall by Algorithm:
  UMKP: 59.00% (59/100 cases)
  AltRB: 60.34% (70/116 cases)
  kPEX: 60.33% (219/363 cases)
  Pivot_Br: 73.45% (166/226 cases)
  S_Br: 67.19% (172/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 74281.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.111
  Avg Overhead/case ((pred - opt) / opt)    = 21.51%


Adding prediction probabilities...

Saving full predictions to './XGB_results/tie_tol/50/expand_and_split/all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 347 wrong predictions with time metrics

T

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [ ]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [49]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [272]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [273]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.0  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [274]:
from xgboost import XGBClassifier

# Train XGBoost with sample weights
xgb_holdout_strat_y = XGBClassifier(
     objective="multi:softprob",
     num_class=len(data_splits.algorithms),      # = 5 in our case
     n_estimators=200,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     use_label_encoder=False,
    #  random_state=42
 )

xgb_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_class=5, ...)

Feature Importances

In [252]:
# --- Feature Importance ---
importances = pd.DataFrame({
    'feature': X_train_.columns,
    'importance': xgb_holdout_strat_y.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
3           d_avg    0.217703
5        core_max    0.148979
6        core_min    0.100083
7       shell_num    0.098166
10  shell_minSize    0.092512
8   shell_avgSize    0.071431
4           d_max    0.061808
0         k_value    0.059812
11  shell_stdSize    0.056510
2               m    0.049917
1               n    0.043081
9   shell_maxSize    0.000000


#### Inference on 20% dataset

In [250]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=xgb_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./XGB_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./XGB_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 55.77%

Precision by Algorithm:
  UMKP: 50.00% (12 predictions)
  AltRB: 50.00% (16 predictions)
  kPEX: 60.00% (50 predictions)
  Pivot_Br: 42.86% (42 predictions)
  S_Br: 69.44% (36 predictions)

Recall by Algorithm:
  UMKP: 55.00% (11/20 cases)
  AltRB: 56.52% (13/23 cases)
  kPEX: 65.75% (48/73 cases)
  Pivot_Br: 66.67% (30/45 cases)
  S_Br: 76.47% (39/51 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 23096.10
  Total Optimal Time   = 16592.90
  Time Ratio (pred/opt)= 1.392
  Avg Overhead/case ((pred - opt) / opt)    = 4400.85%


Adding prediction probabilities...

Saving full predictions to './XGB_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 69 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_val

#### Re-evalute on entire dataset

In [251]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=xgb_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./XGB_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./XGB_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 59.74%

Precision by Algorithm:
  UMKP: 46.03% (63 predictions)
  AltRB: 55.56% (90 predictions)
  kPEX: 71.90% (242 predictions)
  Pivot_Br: 38.34% (193 predictions)
  S_Br: 73.84% (172 predictions)

Recall by Algorithm:
  UMKP: 54.00% (54/100 cases)
  AltRB: 60.34% (70/116 cases)
  kPEX: 67.77% (246/363 cases)
  Pivot_Br: 72.12% (163/226 cases)
  S_Br: 75.39% (193/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 76955.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.151
  Avg Overhead/case ((pred - opt) / opt)    = 36.42%


Adding prediction probabilities...

Saving full predictions to './XGB_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 306 wrong predictions with time metrics

Top 5 

### **Balanced train-test split based on Y and K**

In [275]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [276]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.0  # algorithms within tie_tolerance% of min treated as ties

)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [277]:
# Train XGBoost with sample weights
xgb_holdout_strat_y_K = XGBClassifier(
     objective="multi:softprob",
     num_class=len(data_splits.algorithms),      # = 5 in our case
     n_estimators=150,
     max_depth=15,
     learning_rate=0.75,              # you can tune this later
     use_label_encoder=False,
     random_state=42
 )

xgb_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.75, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=15,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=150,
              n_jobs=None, num_class=5, ...)

Feature Importances

In [278]:
# --- Feature Importance ---
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_holdout_strat_y_K.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
2               m    0.203322
11  shell_stdSize    0.147078
4           d_max    0.130430
3           d_avg    0.108710
7       shell_num    0.089712
5        core_max    0.084306
6        core_min    0.064536
1               n    0.060959
10  shell_minSize    0.058342
0         k_value    0.045556
8   shell_avgSize    0.007049
9   shell_maxSize    0.000000


#### Inference on 20% dataset

In [264]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=xgb_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./XGB_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./XGB_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 56.08%

Precision by Algorithm:
  UMKP: 40.00% (10 predictions)
  AltRB: 52.94% (17 predictions)
  kPEX: 64.41% (59 predictions)
  Pivot_Br: 30.00% (30 predictions)
  S_Br: 71.88% (32 predictions)

Recall by Algorithm:
  UMKP: 55.00% (11/20 cases)
  AltRB: 60.87% (14/23 cases)
  kPEX: 68.49% (50/73 cases)
  Pivot_Br: 60.00% (27/45 cases)
  S_Br: 72.55% (37/51 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 12744.50
  Total Optimal Time   = 11002.60
  Time Ratio (pred/opt)= 1.158
  Avg Overhead/case ((pred - opt) / opt)    = 4550.09%


Adding prediction probabilities...

Saving full predictions to './XGB_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 65 wrong predictions with time metrics

Top 5 heaviest wr

#### Re-evalute on entire dataset

In [265]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=xgb_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./XGB_results/tie_tol/50/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./XGB_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 58.82%

Precision by Algorithm:
  UMKP: 40.00% (65 predictions)
  AltRB: 56.25% (96 predictions)
  kPEX: 68.95% (248 predictions)
  Pivot_Br: 40.00% (185 predictions)
  S_Br: 73.49% (166 predictions)

Recall by Algorithm:
  UMKP: 52.00% (52/100 cases)
  AltRB: 64.66% (75/116 cases)
  kPEX: 64.74% (235/363 cases)
  Pivot_Br: 70.35% (159/226 cases)
  S_Br: 74.61% (191/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 73019.20
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.092
  Avg Overhead/case ((pred - opt) / opt)    = 19.05%


Adding prediction probabilities...

Saving full predictions to './XGB_results/tie_tol/50/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 313 wrong predictions with time metrics

Top 

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br